# Use case — `Utility/time_delay_pspline.py`

Adaptive shared P-spline estimation for two, three or four components. The first component is the zero-delay reference.

**Convention:** `t_shifted_k = t_k - delay_k`. Inspect LOO, overlap and K profiles before interpreting the selected delay.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility import time_delay_pspline as td

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"  # Canonical preprocessed light curves.
SOURCE_ID = "Source_ID"                               # System containing all requested components.
COMPONENT_IDS = [                                                # Two to four IDs; first entry defines delay zero.
    "Component_A_ID",
    "Component_B_ID",
]
COMPONENT_NAMES = ["A reference", "B"]                         # Human-readable labels matching COMPONENT_IDS.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")

## Estimator hyperparameters

This complete dictionary preserves the original `TDPSPL.ipynb` adaptation. For a first run, replace it with `td.load_parameter_profile("quick")`.

In [ ]:
# ============================================================
# P-SPLINE TIME-DELAY ESTIMATION CONFIGURATION
# ============================================================
#
# For a first run, it is recommended to keep these values.
# The parameters most commonly changed are dmin, dmax, ngrid,
# min_points, min_frac, and min_span_frac.
#
# The first light curve passed to the program is always the reference.
# Its delay is fixed at 0 days. The delays of the other light curves are
# calculated relative to this first curve.


estimator_kwargs = {

    # --------------------------------------------------------
    # 1. RANGE IN WHICH THE DELAY IS SEARCHED
    # --------------------------------------------------------

    # Smallest delay that the program is allowed to test.
    # Here, the search starts at -500 days.
    # Decrease this value, for example to -700, to search for
    # larger negative delays.
    "dmin": -500,

    # Largest delay that the program is allowed to test.
    # Here, the search stops at +500 days.
    # Increase this value if a delay greater than 500 days
    # is scientifically possible.
    "dmax": 500,

    # Number of delay values tested between dmin and dmax.
    # A larger value gives a more precise initial search,
    # but greatly increases the computation time.
    # With -500, +500, and 400 values, the spacing is about 2.5 days.
    "ngrid": 400,

    # Number of times the program successively re-optimises
    # the delays of all components.
    # One pass is generally sufficient for two light curves.
    # For three or four curves, using 2 or 3 may be more robust,
    # but it multiplies the computation time.
    "max_delay_passes": 1,

    # Numerical precision of the final refinement, in days.
    # 0.05 days corresponds to approximately 1.2 hours.
    # Warning: this is not the scientific uncertainty of the result.
    "scalar_xatol": 0.05,


    # --------------------------------------------------------
    # 2. SHAPE OF THE P-SPLINE CURVE
    # --------------------------------------------------------

    # Degree of the polynomial pieces forming the spline.
    # 3 corresponds to a cubic spline, which is the standard choice.
    # It is generally not recommended to change this parameter.
    "degree": 3,

    # Legacy parameter used to select the number of spline knots
    # automatically.
    # With K_selection="loo", it normally has very little effect.
    # The value 999 generally forces the legacy rule to propose K_min.
    # It is mainly kept as a fallback solution.
    "points_per_interval": 999,

    # Smallest number of internal knots that the program may test.
    # A small value produces a simpler and smoother curve.
    "K_min": 5,

    # Largest number of internal knots that the program may test.
    # A large value allows a more detailed curve, but increases
    # the risk of fitting noise and increases computation time.
    # The program may automatically use a smaller limit when
    # there are not enough observations.
    "K_max": 150,


    # --------------------------------------------------------
    # 3. AUTOMATIC SELECTION OF THE NUMBER OF KNOTS K
    # --------------------------------------------------------

    # Method used to select the complexity of the spline.
    # "loo" tests how well the spline predicts each observation
    # when that observation is temporarily removed from the model.
    # This is the recommended method.
    "K_selection": "loo",
    
    # List of spline complexities tested by the program.
    #
    # None asks the program to build the list automatically from the
    # number of unique observation times in the current overlap.
    #
    # The automatic list combines:
    # - regularly spaced K values;
    # - additional small K values;
    # - limits imposed by the available number of observations.
    #
    # The list may change slightly for different tested delays because
    # the number of observations in the temporal overlap can change.
    #
    # Example of a manually fixed list:
    # "K_loo_candidates": [5, 10, 20, 40, 80]
    #
    # For normal use, leave this as None.
    "K_loo_candidates": None,

    # Approximate number of K values tested when
    # K_loo_candidates=None.
    # Increasing this value explores more models, but considerably
    # slows down the calculation.
    "K_loo_n_candidates": 15,

    # Penalty added to overly complex splines when selecting K.
    # A larger value favours simpler splines.
    # A smaller value allows more flexible splines.
    "K_loo_complexity_penalty": 1.0,

    # Safety limit against extremely flexible models.
    # A model is rejected if its effective complexity exceeds 80%
    # of the observations available in the overlap.
    # It is recommended to keep this value.
    "K_loo_df_cap_frac": 0.80,


    # --------------------------------------------------------
    # 4. CRITERION USED TO SELECT THE DELAY
    # --------------------------------------------------------

    # Score minimised by the program to find the delay.
    #
    # "loo"  : Leave-One-Out prediction quality; recommended.
    # "bic"  : compromise between goodness of fit and complexity.
    # "chi2" : direct goodness of fit; more sensitive to overfitting.
    #
    # Keep "loo" for the main analysis.
    "delay_score": "loo",


    # --------------------------------------------------------
    # 5. LOCAL ESTIMATION OF NOISE AND VARIABILITY
    # --------------------------------------------------------

    # Requested neighbourhood size used to estimate the local
    # dispersion of the flux values.
    #
    # Important: in the current version, the program generally tries
    # to use at least 5 points. A value of 3 therefore often produces
    # an effective neighbourhood containing approximately 5 points.
    "rolling_window": 3,

    # Method used to build the local neighbourhood.
    #
    # "points": uses neighbouring observations, even when they are
    #           far apart in time.
    # "time"  : uses all observations located within a given time radius.
    #
    # "points" is simple and works even with irregular time sampling.
    "rolling_mode": "points",

    # Time radius, in days, used only when rolling_mode="time".
    #
    # None asks the program to estimate it automatically from the usual
    # spacing between observations.
    # This parameter is ignored when rolling_mode="points".
    "rolling_time_radius": None,

    # True combines the measurement error supplied in the data with
    # the local dispersion observed around each date.
    # This prevents highly variable regions from receiving too much weight.
    #
    # False uses only the measurement errors.
    "use_global_mad_in_sigma": True,

    # Smallest allowed value for errors and dispersions after the
    # light curves have been normalised.
    # This prevents an observation with an almost zero error from
    # dominating the entire calculation.
    "mad_floor": 0.05,


    # --------------------------------------------------------
    # 6. STRENGTH OF THE P-SPLINE SMOOTHING
    # --------------------------------------------------------

    # General smoothing level.
    #
    # "auto" asks the program to calculate it automatically from the
    # dispersion observed in the light curves.
    #
    # A numerical value, for example 1.0, may also be imposed,
    # but "auto" is recommended.
    "lambda_base": "auto",

    # Factor applied to the automatically calculated smoothing level.
    #
    # Increasing this value produces a smoother curve.
    # Decreasing this value produces a more flexible curve.
    #
    # This parameter is used only when lambda_base="auto".
    "lambda_scale": 0.03,

    # Controls how smoothing changes across different time regions.
    #
    # 0.0: identical smoothing everywhere.
    # 0.5: moderate adaptation to local variations.
    # 1.0: stronger adaptation.
    #
    # In this code, a high local dispersion produces stronger local
    # smoothing.
    "lambda_alpha": 0.5,

    # Smallest allowed local penalty, expressed as a fraction of the
    # general lambda_base level.
    #
    # 0.05 means that a region may be smoothed at most 20 times less
    # than the general level.
    "lambda_min_ratio": 0.05,

    # Largest allowed local penalty.
    #
    # 5.0 means that a region may be smoothed at most 5 times more
    # than the general level.
    "lambda_max_ratio": 5.0,


    # --------------------------------------------------------
    # 7. BIC DIAGNOSTICS
    # --------------------------------------------------------
    #
    # With delay_score="loo", the following two parameters do not
    # directly change the final delay. They are mainly used to calculate
    # the BIC diagnostic displayed in the results.

    # Complexity limit used by the BIC diagnostic.
    # A large penalty is applied if the effective complexity exceeds
    # 30% of the number of observations.
    "df_cap_frac": 0.30,

    # Importance assigned to spline curvature in the BIC score.
    # A larger value penalises irregular curves more strongly.
    # With delay_score="loo", simply keep this value at 1.0.
    "rough_penalty": 1.0,


    # --------------------------------------------------------
    # 8. TEMPORAL OVERLAP BETWEEN THE LIGHT CURVES
    # --------------------------------------------------------

    # Additional penalty when the time shift removes a large part
    # of the observations from the common time interval.
    #
    # 0.0 disables this soft penalty.
    # The mandatory min_points, min_frac, and min_span_frac conditions
    # remain active even when this value is zero.
    "overlap_penalty": 0.0,

    # Ideal overlap duration used by the soft penalty.
    # 1.0 represents 100% of the duration of the shortest light curve.
    # This parameter has no effect when overlap_penalty=0.0.
    "min_span_frac_ref": 1.0,

    # Ideal fraction of observations retained in every component.
    # 1.0 represents 100% of the observations.
    # This parameter has no effect when overlap_penalty=0.0.
    "min_frac_ref": 1.0,

    # Minimum number of observations that each component must retain
    # in the common time interval.
    # If even one component retains fewer than 10 points, the tested
    # delay is automatically rejected.
    "min_points": 10,

    # Minimum fraction of observations retained in each light curve.
    # 0.50 means that at least 50% of the points from each component
    # must belong to the common time interval.
    "min_frac": 0.50,

    # Minimum fraction of the temporal duration that must be retained.
    # 0.50 means that the overlap must cover at least 50% of the duration
    # of the shortest light curve.
    #
    # For light curves spanning 10 years, this corresponds approximately
    # to a minimum overlap of 5 years.
    "min_span_frac": 0.50,


    # --------------------------------------------------------
    # 9. SELECTION AND DISPLAY OF THE FINAL RESULT
    # --------------------------------------------------------

    # For two light curves with delay_score="loo", True forces the final
    # delay to correspond exactly to the smallest recorded LOO score,
    # which is also visible on the plot.
    #
    # This parameter is ignored with three or four light curves.
    # If overlap_penalty is later set above zero, it may be preferable
    # to use False so that the total penalised score is respected.
    "force_final_to_min_plotted_loo": True,

    # True displays the progress, estimated delay, score, number of knots,
    # model complexity, and temporal overlap.
    #
    # False performs the same calculations without displaying details.
    "verbose": True,
}

MC_SAMPLES = 300                 # Flux-error draws; use 20 for a smoke test and >=300 for final work.
MC_RANDOM_SEED = 42              # Reproducible random-number seed.
MC_ERROR_SCALE = 1.0             # 1.0 uses flux_obs_error exactly.
MC_FORCE_SAME_K = True           # Reuse base-fit K; False propagates K selection but is much slower.
MC_PROGRESS_EVERY = 10           # Print progress every N draws.

In [ ]:
df = td.load_lightcurve_csv(INPUT_CSV)
system = td.get_components_from_df(
    df=df,
    source_id=SOURCE_ID,
    comp_ids=COMPONENT_IDS,
    names=COMPONENT_NAMES,
)
result = td.estimate_time_delay_pspline(
    system["curves"],
    **estimator_kwargs,
)
td.print_result_multi(system, result)
display(result["pair_delays"])

## Required diagnostics

A delay is not accepted from its scalar value alone. Inspect the objective, LOO, overlap, selected K, fitted curves, rolling-MAD sigma and residuals.

In [ ]:
td.plot_delay_profiles_multi(result, y_col="cost")
td.plot_LOO_score_profile(result)
td.plot_overlap_profile(result)
td.plot_K_profile(result)
td.plot_K_loo_table(result)
td.plot_fit_multi(result, degree=estimator_kwargs["degree"], show_knots=True)
td.plot_global_mad_sigma_diagnostics(result)
td.plot_residuals_multi(result)

## Measurement-error uncertainty

Every draw perturbs each flux by its own `flux_obs_error`, then reruns the estimator. The sample can be multimodal; always inspect its histogram and saved draws.

In [ ]:
uncertainty = td.run_fluxobs_error_mc_pspline(
    system=system,
    estimator_kwargs=estimator_kwargs,
    n_samples=MC_SAMPLES,
    random_seed=MC_RANDOM_SEED,
    error_scale=MC_ERROR_SCALE,
    base_res=result,
    force_same_K_as_base=MC_FORCE_SAME_K,
    progress_every=MC_PROGRESS_EVERY,
    verbose=True,
)
td.print_fluxobs_error_mcmc_uncertainty(uncertainty)
td.plot_fluxobs_error_mcmc_uncertainty(uncertainty, bins=30)
display(uncertainty["pair_delay_summary"])

## Batch command

```bash
python -m Utility.time_delay_pspline data/cleaned_lightcurves.csv --pairs configs/time_delay_system_pairs.csv --profile legacy_notebook --mc-samples 300 --output-dir results/batch_time_delays
```